<a href="https://colab.research.google.com/github/fawkes12345/LNDb-Multimodal/blob/main/notebooks/00_dataset_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LNDb v4 Dataset Audit

## Goal

Understand the structure of LNDb v4 before training any model.

Questions:

1. How many image-annotated nodules are available?
2. How many report-derived nodules are available?
3. How many matched image-report pairs are available?
4. How many image-only and report-only nodules exist?
5. Which semantic attributes are available?

In [1]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/LNDb-Multimodal"
)

DATA_ROOT = PROJECT_ROOT / "data"
METADATA_DIR = DATA_ROOT / "metadata"
CT_DIR = DATA_ROOT / "ct"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project:", PROJECT_ROOT)
print("Metadata:", METADATA_DIR)
print("CT:", CT_DIR)

Project: /content/drive/MyDrive/LNDb-Multimodal
Metadata: /content/drive/MyDrive/LNDb-Multimodal/data/metadata
CT: /content/drive/MyDrive/LNDb-Multimodal/data/ct


In [5]:
for folder in [
    METADATA_DIR,
    CT_DIR,
    CHECKPOINT_DIR,
    OUTPUT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folders ready.")

Folders ready.


In [6]:
import requests
from pathlib import Path

RECORD_ID = "8348419"

FILES = [
    "report.csv",
    "allNods.csv",
    "chars_trainNodules.csv",
    "rad2Fleischner.csv",
    "text2Fleischner.csv",
    "help.txt",
]

host = "zenodo.org"

for filename in FILES:
    url = (
        "https://"
        + host
        + "/records/"
        + RECORD_ID
        + "/files/"
        + filename
    )

    destination = METADATA_DIR / filename

    if destination.exists():
        print(f"Already exists: {filename}")
        continue

    print(f"Downloading {filename}...")

    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(destination, "wb") as f:
        for chunk in response.iter_content(
            chunk_size=8192
        ):
            f.write(chunk)

    print(f"Saved to: {destination}")

Saved to: /content/drive/MyDrive/LNDb-Multimodal/data/metadata/report.csv
Saved to: /content/drive/MyDrive/LNDb-Multimodal/data/metadata/allNods.csv
Saved to: /content/drive/MyDrive/LNDb-Multimodal/data/metadata/chars_trainNodules.csv
Saved to: /content/drive/MyDrive/LNDb-Multimodal/data/metadata/rad2Fleischner.csv
Saved to: /content/drive/MyDrive/LNDb-Multimodal/data/metadata/text2Fleischner.csv
Saved to: /content/drive/MyDrive/LNDb-Multimodal/data/metadata/help.txt


In [7]:
for file in METADATA_DIR.iterdir():
    print(
        file.name,
        round(file.stat().st_size / 1024, 2),
        "KB"
    )

report.csv 23.61 KB
allNods.csv 158.55 KB
chars_trainNodules.csv 68.2 KB
rad2Fleischner.csv 54.27 KB
text2Fleischner.csv 27.97 KB
help.txt 2.29 KB


In [8]:
import pandas as pd

report_df = pd.read_csv(
    METADATA_DIR / "report.csv"
)

all_nodules_df = pd.read_csv(
    METADATA_DIR / "allNods.csv"
)

chars_df = pd.read_csv(
    METADATA_DIR / "chars_trainNodules.csv"
)

rad_df = pd.read_csv(
    METADATA_DIR / "rad2Fleischner.csv"
)

text_df = pd.read_csv(
    METADATA_DIR / "text2Fleischner.csv"
)

In [9]:
display(report_df.head())

,num_report,loc_0,unc_0,rem_0,loc_1,unc_1,rem_1,loc_2,unc_2,rem_2,loc_3,unc_3,rem_3,loc_4,unc_4,rem_4,loc_5,unc_5,rem_5
0,1,RUL,NaN,"apical,8.5,nod,margin: 1",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,"
1,2,LUL,how many?,",,nod,calcification: 5",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,"
2,3,NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,"
3,4,RUL,NaN,",11,nod,internalStructure: 3",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,"
4,5,LUL,how many?,",,micro,",UL,how many?,",,micro,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,",NaN,NaN,",,,"
